# Lean-12b — Théorème de Sensibilité de Huang (companion formel)

Companion **Lean 4** du notebook [Lean-12-Sensitivity-Theorem](Lean-12-Sensitivity-Theorem.ipynb) (Python : *calcule* la sensibilité booléenne $s(f)$ sur l'hypercube et la visualise). Ici, à l'inverse, nous **exhibons la preuve formelle** du théorème de Huang (2019), qui a résolu la *sensitivity conjecture*.

| Voie | Notebook | Méthode |
|------|----------|--------|
| Calcul (existant) | Lean-12-Sensitivity-Theorem (Python) | $s(f)$ numérique, hypercube networkx |
| **Preuve formelle (ce notebook)** | **Lean-12b (companion Lean)** | **Théorème de Huang prouvé en Lean 4, `#check` natif** |

**Lac visité** : [`sensitivity_lean/`](sensitivity_lean/) (lib `Sensitivity`, **0 sorry** — preuve réelle de Hao Huang 2019).

## Convention de vérification — `#check` *natif* via `lake env lean`

> **Mise à niveau SOTA.** Les notebooks companions précédents (c.117–124) vérifiaient l'absence de `sorry` par une *regex indépendante de l'environnement* (verdict `RECOVERABLE-MACHINE` : les oleans Mathlib étant absents, `lake exe cache get` suspendu). **Ce notebook lève cette limite** : il capture les **vraies signatures de théorèmes par `#check` natif** via `lake env lean --stdin`.

**Pourquoi `lake env lean` et non le noyau Jupyter `lean4`/`lean4-wsl` ?** Vérification G.1 directe ce cycle :

- Le lac `sensitivity_lean` **importe Mathlib** (`require mathlib from git`). La **jonction NTFS #2611** relie `.lake/packages/mathlib` au cache partagé `.mathlib-cache/v4.30.0-rc2-1c1dadbc/` qui contient **8182 oleans Mathlib compilés**. Donc `import Sensitivity` *résout* réellement.
- Les noyaux Jupyter `lean4` et `lean4-wsl` lancent un **REPL Lean nu** (sans chemin de recherche des oleans) : `import Sensitivity` échoue (« Unknown identifier »), et la variable d'environnement `LEAN_PATH` est **ignorée**. Seuls les lacs *sans Mathlib* (ex. `lean_game_defs`, companions c.118/119) fonctionnent nativement en noyau.
- En revanche **`lake env lean --stdin` configure automatiquement le chemin des oleans** (lake + jonction Mathlib) : on obtient les **vraies signatures** des théorèmes du lac. C'est le canal de vérification natif *réellement disponible* pour un lac important Mathlib.

In [1]:
import subprocess, os, re, pathlib, textwrap

def find_sensitivity_lean_project():
    """Localise le lake sensitivity_lean (ancres multiples)."""
    candidates = [
        r"c:/dev/CoursIA/MyIA.AI.Notebooks/SymbolicAI/Lean/sensitivity_lean",
        "./sensitivity_lean",
        r"../sensitivity_lean",
    ]
    for c in candidates:
        if os.path.isfile(os.path.join(c, "lakefile.lean")):
            return os.path.abspath(c)
    raise FileNotFoundError("lake sensitivity_lean introuvable")

LAKE = find_sensitivity_lean_project()
print("lake sensitivity_lean :", LAKE)
print("toolchain            :", open(os.path.join(LAKE, "lean-toolchain")).read().strip())

def run(cmd, cwd=LAKE, timeout=300, capture=True):
    """Wrapper subprocess Windows (lake.exe natif)."""
    r = subprocess.run(cmd, cwd=cwd, capture_output=capture, text=True,
                       timeout=timeout, shell=isinstance(cmd, str))
    return r.returncode, (r.stdout or "") + (r.stderr or "")

def run_lake_build(target="Sensitivity"):
    """Build-probe honnete (capture le VRAI exit code)."""
    rc, out = run(["lake", "build", target], timeout=600)
    return rc, out

def run_lean_check(lean_src):
    """VERIFICATION NATIVE : lake env lean --stdin -> vraies signatures #check.

    C'est le canal SOTA : contrairement a une regex 0-sorry, on obtient la
    signature reelle produite par le compilateur Lean pour chaque decl cite.
    """
    r = subprocess.run(["lake", "env", "lean", "--stdin"], cwd=LAKE,
                       input=lean_src, capture_output=True, text=True, timeout=300)
    return r.returncode, (r.stdout or "") + (r.stderr or "")

print("helpers prets.")

lake sensitivity_lean : c:\dev\CoursIA\MyIA.AI.Notebooks\SymbolicAI\Lean\sensitivity_lean
toolchain            : leanprover/lean4:v4.30.0-rc2
helpers prets.


### Build-probe (honnête)

Build de la cible `Sensitivity` (umbrella). Le lac étant gros (Mathlib), on rapporte honnêtement le résultat.

In [2]:
rc, out = run_lake_build("Sensitivity")
tail = "\n".join(out.splitlines()[-15:])
print(f"lake build Sensitivity -> exit={rc}")
print(tail if tail.strip() else "(pas de sortie - deja compile / a travers la jonction #2611)")
print("\nCommande manuelle si besoin :")
print(f"  cd {LAKE} && lake build Sensitivity")

lake build Sensitivity -> exit=0
Build completed successfully (1933 jobs).

Commande manuelle si besoin :
  cd c:\dev\CoursIA\MyIA.AI.Notebooks\SymbolicAI\Lean\sensitivity_lean && lake build Sensitivity


## Vérification native — `#check` des théorèmes piliers

**Cœur de la mise à niveau SOTA** : au lieu d'une regex, on demande au compilateur Lean les **vraies signatures**. On `#check` les quatre piliers :
1. `f_squared` — la relation spectrale $f_n^2(v) = n \cdot v$ (cœur algébrique),
2. `g_injective` — l'injectivité de l'opérateur de Knuth $g_m$,
3. `exists_eigenvalue` — lemme : un grand sous-ensemble $H$ force une valeur propre,
4. **`huang_degree_theorem`** — le **théorème de Huang** (flagship, conjecture résolue).

In [3]:
check_src = "import Sensitivity\n"
for decl in ["Sensitivity.f_squared", "Sensitivity.g_injective",
             "Sensitivity.exists_eigenvalue", "Sensitivity.huang_degree_theorem"]:
    check_src += f"#check {decl}\n"

rc, out = run_lean_check(check_src)
# Filtrer les warnings 'repository has local changes' (jonction, benins)
lines = [l for l in out.splitlines()
         if l.strip() and 'has local changes' not in l and not l.startswith('warning:')]
print(f"lake env lean --stdin -> exit={rc}  ({len(lines)} lignes utiles)\n")
for l in lines:
    print(l)

ok = any('huang_degree_theorem' in l for l in lines) and rc == 0
print("\n=> #check natif REUSSI" if ok else "\n=> verifier la sortie ci-dessus")

lake env lean --stdin -> exit=0  (6 lignes utiles)

Sensitivity.f_squared {n : ℕ} (v : Sensitivity.V n) : (Sensitivity.f n) ((Sensitivity.f n) v) = ↑n • v
Sensitivity.g_injective {m : ℕ} : Function.Injective ⇑(Sensitivity.g m)
Sensitivity.exists_eigenvalue {m : ℕ} (H : Set (Sensitivity.Q m.succ)) (hH : H.toFinset.card ≥ 2 ^ m + 1) :
  ∃ y ∈ Submodule.span ℝ (Sensitivity.e '' H) ⊓ (Sensitivity.g m).range, y ≠ 0
Sensitivity.huang_degree_theorem {m : ℕ} (H : Set (Sensitivity.Q m.succ)) (hH : H.toFinset.card ≥ 2 ^ m + 1) :
  ∃ q ∈ H, √(↑m + 1) ≤ ↑(H ∩ q.adjacent).toFinset.card

=> #check natif REUSSI


## Sorry-check (complément — le lac est 0 sorry)

Double vérification : regex complète (bullet `·` U+00B7 + `exact sorry` + `:= sorry`) sur les 4 modules + umbrella.

In [4]:
SORRY_RE = re.compile(r"^\s*sorry\s*$|:=\s*by\s*sorry|:=\s*sorry\s*$|\bexact\s+sorry\b|^\s*[\u00B7-]\s*sorry\b")
mods = ["Sensitivity/Hypercube.lean", "Sensitivity/VectorSpace.lean",
        "Sensitivity/Operator.lean", "Sensitivity/MainTheorem.lean", "Sensitivity.lean"]
total = 0
for m in mods:
    src = open(os.path.join(LAKE, m), encoding="utf-8").read()
    n = len(SORRY_RE.findall(src))
    total += n
    print(f"  {m:42s} sorry={n}")
print(f"  {'TOTAL':42s} sorry={total}  -> {'OK (0 sorry)' if total == 0 else 'NON-ZERO'}")

  Sensitivity/Hypercube.lean                 sorry=0
  Sensitivity/VectorSpace.lean               sorry=0
  Sensitivity/Operator.lean                  sorry=0
  Sensitivity/MainTheorem.lean               sorry=0
  Sensitivity.lean                           sorry=0
  TOTAL                                      sorry=0  -> OK (0 sorry)


## Arc pédagogique — la stratégie de preuve de Huang

La preuve (Huang 2019) repose sur un argument spectral élégant sur l'hypercube $Q_n$. Nous exhibons les sources réelles des quatre modules, dans l'ordre logique.

### 1. Hypercube $Q_n$ — le graphe-support

$Q_n = \text{Fin}\,n \to \text{Bool}$ : les sommets sont les mots binaires. La projection $\pi$ oublie la première coordonnée. Lemmes : $|Q_n| = 2^n$ (pigeonhole), et la décomposition $Q_{n+1} \simeq Q_n \sqcup Q_n$.

In [5]:
def show(mod, lo=None):
    p = os.path.join(LAKE, mod)
    lines = open(p, encoding="utf-8").read().splitlines()
    if lo: lines = lines[lo[0]-1:lo[1]]
    print(f"--- {mod} ({len(lines)} lignes) ---")
    for l in lines:
        print(l)

show("Sensitivity/Hypercube.lean", (36, 62))

--- Sensitivity/Hypercube.lean (27 lignes) ---
/-- The hypercube in dimension `n`. -/
abbrev Q (n : ℕ) :=
  Fin n → Bool

/-- The projection from `Q n.succ` to `Q n` forgetting the first value
(i.e. the image of zero). -/
def π {n : ℕ} : Q n.succ → Q n := fun p => p ∘ Fin.succ

namespace Q

@[ext]
theorem ext {n} {f g : Q n} (h : ∀ x, f x = g x) : f = g := funext h

variable (n : ℕ)

/-- `Q 0` has a unique element. -/
instance : Unique (Q 0) :=
  ⟨⟨fun _ => true⟩, by intro; ext x; fin_cases x⟩

/-- `Q n` has 2^n elements. -/
theorem card : Fintype.card (Q n) = 2 ^ n := by simp

variable {n}

theorem succ_n_eq (p q : Q n.succ) : p = q ↔ p 0 = q 0 ∧ π p = π q := by
  constructor
  · rintro rfl; exact ⟨rfl, rfl⟩


### 2. Espace vectoriel libre $V_n$ — l'algèbre linéaire

$V_n$ = espace vectoriel libre sur les sommets de $Q_n$ (défini inductivement : $V_0 = \mathbb{R}$, $V_{n+1} = V_n \times V_n$). $e$ est la base canonique, $\varepsilon$ la base duale. Le lemme de **dualité** $\varepsilon_p(e_q) = [p=q]$ fonde toute l'algèbre.

In [6]:
show("Sensitivity/VectorSpace.lean", (33, 80))

--- Sensitivity/VectorSpace.lean (48 lignes) ---
/-- The free vector space on vertices of a hypercube, defined inductively. -/
def V : ℕ → Type
  | 0 => ℝ
  | n + 1 => V n × V n

@[simp]
theorem V_zero : V 0 = ℝ := rfl

@[simp]
theorem V_succ {n : ℕ} : V (n + 1) = (V n × V n) := rfl

namespace V

@[ext]
theorem ext {n : ℕ} {p q : V n.succ} (h₁ : p.1 = q.1) (h₂ : p.2 = q.2) : p = q := Prod.ext h₁ h₂

variable (n : ℕ)

instance : DecidableEq (V n) := by induction n <;> · dsimp only [V]; infer_instance

instance : AddCommGroup (V n) := by induction n <;> · dsimp only [V]; infer_instance

set_option backward.isDefEq.respectTransparency false in
instance : Module ℝ (V n) := by induction n <;> · dsimp only [V]; infer_instance

end V

/-- The basis of `V` indexed by the hypercube, defined inductively. -/
noncomputable def e : ∀ {n}, Q n → V n
  | 0 => fun _ => (1 : ℝ)
  | Nat.succ _ => fun x => cond (x 0) (e (π x), 0) (0, e (π x))

@[simp]
theorem e_zero_apply (x : Q 0) : e x = (1 : ℝ) :=
  r

### 3. Opérateur spectral $f_n$ — le cœur algébrique

L'opérateur linéaire $f_n : V_n \to V_n$ (matrice $A_n$ de Huang), défini inductivement. **Lemme central** : $f_n^2 = n \cdot \mathrm{Id}$ (`f_squared`) — $f_n$ est donc « presque » une racine carrée de $n$. L'opérateur de Knuth $g_m$ satisfait $f_{m+1}(g_m v) = \sqrt{m+1}\, v$ (`f_image_g`).

In [7]:
show("Sensitivity/Operator.lean", (32, 92))

--- Sensitivity/Operator.lean (61 lignes) ---
/-- The linear operator $f_n$ corresponding to Huang's matrix $A_n$,
defined inductively as a ℝ-linear map from `V n` to `V n`. -/
noncomputable def f : ∀ n, V n →ₗ[ℝ] V n
  | 0 => 0
  | n + 1 =>
    LinearMap.prod (LinearMap.coprod (f n) LinearMap.id) (LinearMap.coprod LinearMap.id (-f n))

@[simp]
theorem f_zero : f 0 = 0 :=
  rfl

theorem f_succ_apply (v : V n.succ) : f n.succ v = (f n v.1 + v.2, v.1 - f n v.2) := by
  cases v
  rw [f]
  simp only [sub_eq_add_neg]
  exact rfl

set_option backward.isDefEq.respectTransparency false in
theorem f_squared (v : V n) : (f n) (f n v) = (n : ℝ) • v := by
  induction n with
  | zero => simp only [Nat.cast_zero, zero_smul, f_zero, zero_apply]
  | succ n IH =>
    cases v; rw [f_succ_apply, f_succ_apply]; simp [IH, add_smul (n : ℝ) 1, add_assoc]; abel

set_option backward.isDefEq.respectTransparency false in
open Classical in
theorem f_matrix (p q : Q n) : |ε q (f n (e p))| = if p ∈ q.adjacent then 

### 4. Théorème principal — Huang 2019

**Flagship `huang_degree_theorem`** : tout sous-ensemble $H \subseteq Q_{m+1}$ avec $|H| \geq 2^m + 1$ contient un sommet $q$ ayant au moins $\sqrt{m+1}$ voisins dans $H$.

**Corollaire** (conjecture de sensibilité) : pour toute fonction booléenne $f$ sur $n$ bits, $s(f) \geq \sqrt{n}$ (prendre $H$ = un sous-cube de sensibilité minimale). C'est exactement ce que le notebook Python Lean-12 *mesure numériquement* ; ici il est *prouvé*.

La preuve combine `exists_eigenvalue` (un grand $H$ force une valeur propre $\sqrt{m+1}$ via $g_m$ injectif) puis un argument de restriction.

In [8]:
show("Sensitivity/MainTheorem.lean", (48, 95))

--- Sensitivity/MainTheorem.lean (48 lignes) ---
/-- If a subset `H` of `Q (m+1)` has cardinal at least `2^m + 1` then the
subspace of `V (m+1)` spanned by the corresponding basis vectors non-trivially
intersects the range of `g m`. -/
theorem exists_eigenvalue (H : Set (Q m.succ)) (hH : Card H ≥ 2 ^ m + 1) :
    ∃ y ∈ Span (e '' H) ⊓ range (g m), y ≠ 0 := by
  let W := Span (e '' H)
  let img := range (g m)
  suffices 0 < dim (W ⊓ img) by
    exact mod_cast exists_mem_ne_zero_of_rank_pos this
  have dim_le : dim (W ⊔ img) ≤ 2 ^ (m + 1 : Cardinal) := by
    convert ← Submodule.rank_le (W ⊔ img)
    rw [← Nat.cast_succ]
    apply dim_V
  have dim_add : dim (W ⊔ img) + dim (W ⊓ img) = dim W + 2 ^ m := by
    convert ← Submodule.rank_sup_add_rank_inf_eq W img
    rw [rank_range_of_injective (g m) g_injective]
    apply dim_V
  have dimW : dim W = Fintype.card H := by
    have li : LinearIndependent ℝ (H.restrict e) := by
      convert (dualBases_e_ε m.succ).basis.linearIndependent.comp _ 

## Conclusion

| Élément | Détail |
|---------|--------|
| Lac | `sensitivity_lean` (lib `Sensitivity`) |
| Preuve | Hao Huang 2019 — résout la *sensitivity conjecture* |
| `sorry` | **0** sur les 4 modules + umbrella (regex complète) |
| **Vérification native** | **`#check` réel via `lake env lean --stdin`** (signatures du compilateur) — *upgrade SOTA vs regex* |
| Why not lean4 kernel | REPL nu : ne peut pas importer Mathlib (G.1 vérifié ce cycle) |

**Jalon ouvert documenté** : la borne fine $s(f) \geq \sqrt{n}$ comme *corollaire explicite* (réduction combinatoire H→sous-cube) n'est pas formalisée dans le lac — seul le théorème de degré l'est. La lib reste sorry-free.

## Exercices

**Exercice 1 (Python).** Calculez la sensibilité $s(f)$ de la fonction majorité sur $n=3$ bits (comme Lean-12) et vérifiez qu'elle respecte la borne $s(f) \geq \lceil\sqrt{3}\rceil = 2$.

In [9]:
from itertools import product

def sensitivity(f, n):
    # TODO etudiant : renvoyer max_x |{i : f(x) != f(x flipped i)}|
    # return max(sum(f(x) != f(x[:i]+(1-x[i],)+x[i+1:]) for i in range(n)) for x in product([0,1],repeat=n))
    return 0  # a completer

majority3 = lambda x: 1 if sum(x) >= 2 else 0
print("s(majorite, 3) =", sensitivity(majority3, 3), "(borne >= 2)")

s(majorite, 3) = 0 (borne >= 2)


**Exercice 2 (Lean).** En vous basant sur `f_squared`, montrez que $f_n$ ne peut pas être nilpotent (indication : appliquez $f_n^2$ à un vecteur non nul). *Stub à compléter.*

In [10]:
# Exercice 2 (Lean) -- a completer dans un REPL lake env lean
# example : forall (n : Nat) (v : Sensitivity.V n), n > 0 -> v != 0 ->
#           Sensitivity.f n (Sensitivity.f n v) != 0 := by
#   -- Indice : Sensitivity.f_squared dit f (f v) = n • v ; n > 0 et v != 0
#   sorry
print("Exercice 2 : preuve Lean a completer (cf f_squared)")

Exercice 2 : preuve Lean a completer (cf f_squared)


**Exercice 3 (Python).** Vérifiez numériquement la borne $s(f) \geq \sqrt{n}$ pour *toutes* les fonctions booléennes sur $n=4$ bits (il y en a $2^{2^4}=65536$) : aucune ne descend sous $\sqrt{4}=2$.

In [11]:
from math import sqrt, isqrt
from itertools import product

def s_min_all_functions(n):
    # TODO etudiant : enumerer les 2^(2^n) fonctions, calculer s(f) min
    # assert min >= isqrt(n) or n == 0
    return None  # a completer

# Attention : n=4 = 65536 fonctions, reste rapide (< 1 s)
print("Exercice 3 : s_min sur n=4 a calculer (attendu >= 2)")

Exercice 3 : s_min sur n=4 a calculer (attendu >= 2)
